<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/VQ_VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lpips

In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time
import numpy as np

In [ ]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
class Residual(nn.Module):
    def __init__(self, num_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(num_channels, num_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return x + self.block(x)

class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=64, num_channels=3):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1),                      # 512 -> 256
            nn.ReLU(),
            nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1),             # 256 -> 128
            nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1),              # 128 -> 64
            nn.ReLU(),
            nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1),          # 64 -> 32
            nn.ReLU(),
            nn.Conv2d(hidden_dim * 4, embedding_dim, kernel_size=4, stride=2, padding=1),           # 32 -> 16
        )
        self.residual = Residual(embedding_dim)

    def forward(self, x):
        x = self.conv(x)
        return self.residual(x)

class Decoder(nn.Module):
    def __init__(self, embedding_dim=64, hidden_dim=128, num_channels=3):
        super().__init__()

        self.residual = Residual(embedding_dim)
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, hidden_dim * 4, kernel_size=4, stride=2, padding=1),                      # 512 -> 256
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim * 4, hidden_dim * 2, kernel_size=4, stride=2, padding=1),             # 256 -> 128
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1),              # 128 -> 64
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1),          # 64 -> 32
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim // 2, num_channels, kernel_size=4, stride=2, padding=1),           # 32 -> 16
        )

    def forward(self, x):
        x = self.residual(x)
        x = self.conv(x)
        return torch.tanh(x)

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=64, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        return quantized, loss, perplexity

class VQ_VAE(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=64, hidden_dim=128, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, perplexity = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity


In [ ]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VQ_VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.9))

In [ ]:
model.train()

epochs = 20

for epoch in range(epochs):
    epoch_start = time.time()
    total_loss, epoch_perplexity = 0.0, 0.0

    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        data_recon, vq_loss, perplexity = model(data)
        recon_loss = F.mse_loss(data_recon, data)
        loss = recon_loss + vq_loss
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        epoch_perplexity += perplexity.item()

        if batch_idx % 100 == 0:
            print(f'Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')

    print(f'====> Epoch: {epoch} | Average loss: {total_loss / len(train_loader):.4f} | Perplexity: {epoch_perplexity/len(train_loader):.2f}')

    print(f"Epoch {epoch} computed in ", time.time() - epoch_start ,"sec")



Epoch 0 [0/1880] Loss: 0.266219
====> Epoch: 0 | Average loss: 88.6142 | Perplexity: 3.61
Epoch 0 computed in  357.9358286857605 sec
Epoch 1 [0/1880] Loss: 43.349388
====> Epoch: 1 | Average loss: 19.3713 | Perplexity: 2.98
Epoch 1 computed in  58.44019794464111 sec
Epoch 2 [0/1880] Loss: 6.021046
====> Epoch: 2 | Average loss: 6.0653 | Perplexity: 3.21
Epoch 2 computed in  57.85861015319824 sec
Epoch 3 [0/1880] Loss: 16.489553
====> Epoch: 3 | Average loss: 26.9766 | Perplexity: 3.31
Epoch 3 computed in  57.83227229118347 sec
Epoch 4 [0/1880] Loss: 17.123203
====> Epoch: 4 | Average loss: 16.1769 | Perplexity: 3.69
Epoch 4 computed in  58.134923696517944 sec
Epoch 5 [0/1880] Loss: 42.257462
====> Epoch: 5 | Average loss: 17.0445 | Perplexity: 3.92
Epoch 5 computed in  57.83888649940491 sec
Epoch 6 [0/1880] Loss: 25.405973
====> Epoch: 6 | Average loss: 26.7186 | Perplexity: 4.52
Epoch 6 computed in  58.04428029060364 sec
Epoch 7 [0/1880] Loss: 12.927339
====> Epoch: 7 | Average loss: 